# Viva-munk Composite Gallery

_Investigation `composite-gallery` — coder reproduction notebook._

**Question.** Can every viva-munk composite from the published report be reproduced as an individual,
runnable study in the dashboard, with its simulation GIF shown on the report?

A gallery of the nine viva-munk multi-cell composites, each run as its own study with a
simulation animation and a short biological write-up.

---

**How to use this notebook**

1. **Run ▸ Run All Cells** to reproduce every study end to end.
2. To explore, edit a study's **Parameters** cell, then re-run from its **Run** section — the figures regenerate with your values.
3. The **Setup** cell at the top is collapsed scaffolding (environment + render helpers); expand it only to inspect. Set `RERUN = False` there to render the committed `runs.db` without re-simulating.

The prose states the **question and parameters**; the figures each run produces are the results.


In [ ]:
"""Setup — environment & helpers (runs automatically; open only to inspect).

Generated by vivarium-workbench (notebook_export). This cell resolves the repo,
registers the workspace Core, and defines the render helpers; the science lives
in the sections below. Each study is re-run live with the workspace's own
process-bigraph protocol and its figures rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-munk-studies/viva-munk-studies').is_dir():
    REPO = Path('/home/runner/work/viva-munk-studies/viva-munk-studies')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# Keep the output clean: composite discovery and the pydantic models emit
# benign UserWarnings (optional packages it skips, a field-name shadow) and the
# simulators emit DeprecationWarnings — none are actionable from a reproduction
# notebook, so silence those two categories. Remove this to see them.
import warnings as _warnings
_warnings.filterwarnings("ignore", category=UserWarning)
_warnings.filterwarnings("ignore", category=DeprecationWarning)

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_munk.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

def materialize_composite(spec_id, overrides=None):
    """Build a composite into an editable payload {name, description,
    parameters, state, _overrides, _id}. Works for @composite_generator
    functions via the process-bigraph registry (the common case); ``state``
    is the built document (process nodes at top level)."""
    import importlib
    from process_bigraph.composite_generator import (
        _REGISTRY, apply_core_extensions, build_generator)
    try:
        from process_bigraph.composite_generator import discover_generators
        discover_generators()
    except Exception:
        pass
    entry = _REGISTRY.get(spec_id)
    if entry is None:
        # editable installs are invisible to packages_distributions, so
        # discover alone can miss them — prime the module the id names.
        try:
            importlib.import_module(spec_id.rsplit('.', 1)[0])
            from process_bigraph.composite_generator import discover_generators as _dg
            _dg()
        except Exception:
            pass
        entry = _REGISTRY.get(spec_id)
    if entry is None:
        raise KeyError('composite not registered: ' + repr(spec_id))
    params = dict(entry.parameters or {})
    ov = {k: (v.get('default') if isinstance(v, dict) else None)
          for k, v in params.items()}
    if overrides:
        ov.update({k: val for k, val in overrides.items() if k in params})
    _c = apply_core_extensions(entry, core)
    try:
        doc = build_generator(entry, ov or None, core=_c)
    except TypeError:
        doc = entry.func(_c, **(ov or {}))
    return {'name': entry.name, 'description': entry.description,
            'parameters': params, 'state': doc, '_overrides': ov, '_id': spec_id}

import base64 as _b64, pathlib as _pl
_STATIC_VIZ_SCHEMES = ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:')
def _render_viz_process(address, config):
    """Render a Visualization process/step (e.g. ``local:<Class>``) to
    HTML: wrap it as a one-step composite and run it once — the viz writes
    Plotly HTML to its ``html`` output port. Mirrors the workbench's own
    env-worker viz path. Self-running vizes need no run data."""
    from process_bigraph import Composite
    _doc = {
        'inputs_store': {}, 'output_store': '',
        'visualization': {
            '_type': 'step', 'address': address,
            'config': {k: v for k, v in (config or {}).items()
                       if k not in ('inputs_map', 'sources')},
            'inputs': {}, 'outputs': {'html': ['output_store']},
        },
    }
    _comp = Composite({'state': _doc}, core=core)
    _comp.run(1)
    _html = _comp.state.get('output_store')
    if isinstance(_html, dict):
        _html = _html.get('value') or _html.get('_value') or ''
    return _html if isinstance(_html, str) else ''
def _render_one(address, config, runs_db, study_yaml, overrides=None):
    """Generic figure renderer (no workspace render_study_viz.py).
    A static ``image:``/``file:`` (or bare-path) address embeds the file;
    any other address (``local:<VizProcess>``) is run as a Visualization
    and its HTML returned. ``overrides`` (the composite's edited parameter
    dict) is merged into the viz config first so a self-running figure
    reflects edited parameters; a viz that does not accept those keys
    falls back to the base config."""
    addr = str(address or '')
    _head = addr.split('/', 1)[0]
    _is_static = (not addr) or (':' not in _head) or addr.startswith(_STATIC_VIZ_SCHEMES)
    if not _is_static:
        if core is None:
            return f'<p style="color:#b91c1c">no core to render {address}</p>'
        _base = {k: v for k, v in (config or {}).items()}
        _tries = ([{**_base, **overrides}] if overrides else []) + [_base]
        _err = ''
        for _cfg in _tries:
            try:
                _h = _render_viz_process(addr, _cfg)
            except Exception as _e:
                _err = (f'<p style="color:#b91c1c">visualization {address} not renderable yet: '
                        f'{type(_e).__name__}: {_e}</p>')
                continue
            return _h or f'<p style="color:#6b7280">{address} produced no HTML</p>'
        return _err
    for _scheme in _STATIC_VIZ_SCHEMES:
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Attachment (`attachment`)

**Question.** Do cells settle onto a surface, accumulate adhesins, and secrete EPS once attached?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `attachment` | `viva_munk.composites.attachment` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.attachment`** — `spec_viva_munk_composites_attachment` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_attachment = materialize_composite('viva_munk.composites.attachment')
describe_spec(spec_viva_munk_composites_attachment)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_attachment maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_attachment = dict(spec_viva_munk_composites_attachment['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_attachment, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_attachment['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: attachment ===
STUDY = 'attachment'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_attachment = 100
INTERVAL_attachment = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_attachment if 'ov_viva_munk_composites_attachment' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_attachment = materialize_composite('viva_munk.composites.attachment', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_attachment['state']}, core=core)
            _comp.run(STEPS_attachment)
        print('ran', 'attachment', f'(viva_munk.composites.attachment) for', STEPS_attachment, 'steps')
    except Exception as _e:
        print(f"attachment not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Bending Pressure (`bending-pressure`)

**Question.** Does mechanical pressure from neighbours inhibit the growth of flexible rod-like cells?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `bending_pressure` | `viva_munk.composites.bending_pressure` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.bending_pressure`** — `spec_viva_munk_composites_bending_pressure` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_bending_pressure = materialize_composite('viva_munk.composites.bending_pressure')
describe_spec(spec_viva_munk_composites_bending_pressure)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_bending_pressure maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_bending_pressure = dict(spec_viva_munk_composites_bending_pressure['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_bending_pressure, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_bending_pressure['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: bending-pressure ===
STUDY = 'bending-pressure'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_bending_pressure = 100
INTERVAL_bending_pressure = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_bending_pressure if 'ov_viva_munk_composites_bending_pressure' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_bending_pressure = materialize_composite('viva_munk.composites.bending_pressure', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_bending_pressure['state']}, core=core)
            _comp.run(STEPS_bending_pressure)
        print('ran', 'bending_pressure', f'(viva_munk.composites.bending_pressure) for', STEPS_bending_pressure, 'steps')
    except Exception as _e:
        print(f"bending_pressure not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Biofilm (`biofilm`)

**Question.** How does a colony develop under combined growth, EPS, and pressure dynamics?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `biofilm` | `viva_munk.composites.biofilm` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.biofilm`** — `spec_viva_munk_composites_biofilm` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_biofilm = materialize_composite('viva_munk.composites.biofilm')
describe_spec(spec_viva_munk_composites_biofilm)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_biofilm maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_biofilm = dict(spec_viva_munk_composites_biofilm['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_biofilm, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_biofilm['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: biofilm ===
STUDY = 'biofilm'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_biofilm = 100
INTERVAL_biofilm = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_biofilm if 'ov_viva_munk_composites_biofilm' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_biofilm = materialize_composite('viva_munk.composites.biofilm', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_biofilm['state']}, core=core)
            _comp.run(STEPS_biofilm)
        print('ran', 'biofilm', f'(viva_munk.composites.biofilm) for', STEPS_biofilm, 'steps')
    except Exception as _e:
        print(f"biofilm not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Chemotaxis (`chemotaxis`)

**Question.** Do motile cells climb a static 2D ligand gradient via run-and-tumble?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `chemotaxis` | `viva_munk.composites.chemotaxis` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.chemotaxis`** — `spec_viva_munk_composites_chemotaxis` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_chemotaxis = materialize_composite('viva_munk.composites.chemotaxis')
describe_spec(spec_viva_munk_composites_chemotaxis)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_chemotaxis maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_chemotaxis = dict(spec_viva_munk_composites_chemotaxis['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_chemotaxis, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_chemotaxis['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: chemotaxis ===
STUDY = 'chemotaxis'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_chemotaxis = 100
INTERVAL_chemotaxis = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_chemotaxis if 'ov_viva_munk_composites_chemotaxis' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_chemotaxis = materialize_composite('viva_munk.composites.chemotaxis', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_chemotaxis['state']}, core=core)
            _comp.run(STEPS_chemotaxis)
        print('ran', 'chemotaxis', f'(viva_munk.composites.chemotaxis) for', STEPS_chemotaxis, 'steps')
    except Exception as _e:
        print(f"chemotaxis not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Daughter Machine (`daughter-machine`)

**Question.** How do daughter cells track through an open chamber with an absorbing boundary?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `daughter_machine` | `viva_munk.composites.daughter_machine` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.daughter_machine`** — `spec_viva_munk_composites_daughter_machine` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_daughter_machine = materialize_composite('viva_munk.composites.daughter_machine')
describe_spec(spec_viva_munk_composites_daughter_machine)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_daughter_machine maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_daughter_machine = dict(spec_viva_munk_composites_daughter_machine['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_daughter_machine, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_daughter_machine['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: daughter-machine ===
STUDY = 'daughter-machine'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_daughter_machine = 100
INTERVAL_daughter_machine = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_daughter_machine if 'ov_viva_munk_composites_daughter_machine' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_daughter_machine = materialize_composite('viva_munk.composites.daughter_machine', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_daughter_machine['state']}, core=core)
            _comp.run(STEPS_daughter_machine)
        print('ran', 'daughter_machine', f'(viva_munk.composites.daughter_machine) for', STEPS_daughter_machine, 'steps')
    except Exception as _e:
        print(f"daughter_machine not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Glucose Growth (`glucose-growth`)

**Question.** Does Monod-kinetic growth deplete a diffusing glucose field around the colony?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `glucose_growth` | `viva_munk.composites.glucose_growth` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.glucose_growth`** — `spec_viva_munk_composites_glucose_growth` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_glucose_growth = materialize_composite('viva_munk.composites.glucose_growth')
describe_spec(spec_viva_munk_composites_glucose_growth)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_glucose_growth maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_glucose_growth = dict(spec_viva_munk_composites_glucose_growth['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_glucose_growth, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_glucose_growth['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: glucose-growth ===
STUDY = 'glucose-growth'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_glucose_growth = 100
INTERVAL_glucose_growth = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_glucose_growth if 'ov_viva_munk_composites_glucose_growth' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_glucose_growth = materialize_composite('viva_munk.composites.glucose_growth', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_glucose_growth['state']}, core=core)
            _comp.run(STEPS_glucose_growth)
        print('ran', 'glucose_growth', f'(viva_munk.composites.glucose_growth) for', STEPS_glucose_growth, 'steps')
    except Exception as _e:
        print(f"glucose_growth not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Inclusion Bodies (`inclusion-bodies`)

**Question.** How do inclusion bodies accumulate, burden growth, and segregate across a colony?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `inclusion_bodies` | `viva_munk.composites.inclusion_bodies` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.inclusion_bodies`** — `spec_viva_munk_composites_inclusion_bodies` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_inclusion_bodies = materialize_composite('viva_munk.composites.inclusion_bodies')
describe_spec(spec_viva_munk_composites_inclusion_bodies)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_inclusion_bodies maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_inclusion_bodies = dict(spec_viva_munk_composites_inclusion_bodies['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_inclusion_bodies, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_inclusion_bodies['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: inclusion-bodies ===
STUDY = 'inclusion-bodies'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_inclusion_bodies = 100
INTERVAL_inclusion_bodies = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_inclusion_bodies if 'ov_viva_munk_composites_inclusion_bodies' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_inclusion_bodies = materialize_composite('viva_munk.composites.inclusion_bodies', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_inclusion_bodies['state']}, core=core)
            _comp.run(STEPS_inclusion_bodies)
        print('ran', 'inclusion_bodies', f'(viva_munk.composites.inclusion_bodies) for', STEPS_inclusion_bodies, 'steps')
    except Exception as _e:
        print(f"inclusion_bodies not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Mother Machine (`mother-machine`)

**Question.** How do cells grow when trapped one-per-channel in a mother-machine device?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `mother_machine` | `viva_munk.composites.mother_machine` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.mother_machine`** — `spec_viva_munk_composites_mother_machine` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_mother_machine = materialize_composite('viva_munk.composites.mother_machine')
describe_spec(spec_viva_munk_composites_mother_machine)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_mother_machine maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_mother_machine = dict(spec_viva_munk_composites_mother_machine['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_mother_machine, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_mother_machine['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: mother-machine ===
STUDY = 'mother-machine'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_mother_machine = 100
INTERVAL_mother_machine = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_mother_machine if 'ov_viva_munk_composites_mother_machine' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_mother_machine = materialize_composite('viva_munk.composites.mother_machine', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_mother_machine['state']}, core=core)
            _comp.run(STEPS_mother_machine)
        print('ran', 'mother_machine', f'(viva_munk.composites.mother_machine) for', STEPS_mother_machine, 'steps')
    except Exception as _e:
        print(f"mother_machine not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Study: Quorum Sensing (`quorum-sensing`)

**Question.** Does autoinducer feedback switch a dense colony ON via Hill activation?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `quorum_sensing` | `viva_munk.composites.quorum_sensing` | 0 | — |


### The model & its parameters

Each composite is a process-bigraph *document* — named processes wired by `inputs`/`outputs` ports over shared stores. Below, each composite shows what it does, then a **parameter cell** with its tunable knobs (the current default and what each one means). Edit any value, then re-run the **Run** section to rebuild and re-simulate with your changes.


**Composite `viva_munk.composites.quorum_sensing`** — `spec_viva_munk_composites_quorum_sensing` (materialized from its `@composite_generator`)


In [ ]:
spec_viva_munk_composites_quorum_sensing = materialize_composite('viva_munk.composites.quorum_sensing')
describe_spec(spec_viva_munk_composites_quorum_sensing)

In [ ]:
# === Edit parameters for this composite, then run the Run cell ===
# ov_viva_munk_composites_quorum_sensing maps each tunable parameter to its current value (the generator's
# defaults). Edit any entry; the Run cell rebuilds the composite with it.
ov_viva_munk_composites_quorum_sensing = dict(spec_viva_munk_composites_quorum_sensing['_overrides'])
import json as _json; print(_json.dumps(ov_viva_munk_composites_quorum_sensing, indent=2, default=str))
# e.g.:  ov_viva_munk_composites_quorum_sensing['<parameter>'] = <value>

### Run

_Set `STEPS`, then run. Each composite is built live from its (edited) spec above and run; a composite whose module processes aren't implemented yet reports the error instead of crashing. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: quorum-sensing ===
STUDY = 'quorum-sensing'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

# This study has no *recorded* runs yet. RERUN builds its declared
# composite(s) live and runs them; a composite whose module processes
# are not implemented yet reports the error instead of crashing.
STEPS_quorum_sensing = 100
INTERVAL_quorum_sensing = 1.0

if RERUN:
    from process_bigraph import Composite
    try:
        with quiet():  # keep per-step stdout out of the notebook
            _ov = ov_viva_munk_composites_quorum_sensing if 'ov_viva_munk_composites_quorum_sensing' in dir() else None  # edits from the panel cell, if any
            spec_viva_munk_composites_quorum_sensing = materialize_composite('viva_munk.composites.quorum_sensing', _ov)
            _comp = Composite({'state': spec_viva_munk_composites_quorum_sensing['state']}, core=core)
            _comp.run(STEPS_quorum_sensing)
        print('ran', 'quorum_sensing', f'(viva_munk.composites.quorum_sensing) for', STEPS_quorum_sensing, 'steps')
    except Exception as _e:
        print(f"quorum_sensing not runnable yet: {type(_e).__name__}: {_e}")
else:
    print("RERUN=False — not re-simulating.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**multibody**


In [ ]:
# multibody
show_viz(_render_one('', {}, RUNS_DB, STUDY_YAML))

## Open decisions
- Which composites need parameter calibration against literature before quantitative claims?
- Which readouts should become pass/fail behavior tests?
